# Experiments 45-46 — Quarters and Eighths, Clustered Alone

Same "internal structure" question as three much earlier experiments
in this project (there called Experiments 15, 16, 17), which clustered
the Quran's 60 hizb, 30 hizb-pairs, and 114 chapters with no poets
present at all. This extends that same question to the two finer
divisions introduced more recently:

- **45:** the 240 hizb-quarters, clustered by themselves
- **46:** the 480 eighths (each quarter split in half by verse order),
  clustered by themselves

**No poets, no `poems.db` needed this time** — this notebook only
touches Quran text, so the recurring "wrong folder" issue with the
poetry database simply doesn't apply here.

Reuses your cached Quran text and ayah embeddings if present in this
folder from earlier notebooks — should run fast either way, and even
a cold run only needs the (one-time, ~15-25 min) ayah-embedding step,
not anything poet-related.

Run cells top to bottom, **Shift+Enter**.

In [ ]:
# CELL 1 -- Install packages (lightweight -- no poems.db, no poet embeddings needed)
!pip -q install sentence-transformers torch scikit-learn umap-learn hdbscan pandas numpy matplotlib seaborn requests

In [ ]:
# CELL 2 -- Configuration
import re, json, warnings, pickle
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

warnings.filterwarnings("ignore")

CACHE_DIR = Path("quran_cache"); CACHE_DIR.mkdir(exist_ok=True)
EMBED_CACHE_DIR = Path("embed_cache"); EMBED_CACHE_DIR.mkdir(exist_ok=True)
FIGURES_DIR = Path("output/figures"); FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = Path("output/tables"); TABLES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = Path("output/reports"); REPORTS_DIR.mkdir(parents=True, exist_ok=True)

SBERT_MODEL_NAME = "akhooli/Arabic-SBERT-100K"

UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_N_COMPONENTS_HIGH = 50
UMAP_METRIC = "cosine"
HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = 3

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
import torch
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    print("GPU available:", torch.cuda.get_device_name(0))
else:
    print("No GPU found -- will run on CPU.")

def normalize_word(w):
    w = re.sub(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED\u0670]", "", w)
    w = re.sub(r"\u0640", "", w)
    w = re.sub(r"[\u0622\u0623\u0625\u0671]", "\u0627", w)
    return w.strip()

plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 11})
print("Config loaded.")

In [ ]:
# CELL 3 -- Fetch Quran text WITH hizb-quarter metadata
cache_file = CACHE_DIR / "quran_ayat_with_hizb.json"

if cache_file.exists():
    print("Loading Quran (with hizb metadata) from local cache...")
    with open(cache_file, encoding="utf-8") as f:
        surahs_raw = json.load(f)
else:
    print("Fetching Quran from Al Quran Cloud API...")
    resp = requests.get("https://api.alquran.cloud/v1/quran/quran-uthmani", timeout=60)
    resp.raise_for_status()
    surahs_raw = resp.json()["data"]["surahs"]
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(surahs_raw, f, ensure_ascii=False)
    print("Fetched and cached.")

surah_names = {}
ayah_records = []
for s in surahs_raw:
    snum = s["number"]
    surah_names[snum] = s["englishName"]
    for a in s["ayahs"]:
        hizb_number = ((a["hizbQuarter"] - 1) // 4) + 1
        ayah_records.append({
            "surah_number": snum, "ayah_number": a["numberInSurah"],
            "text": a["text"], "hizb_number": hizb_number,
            "hizb_quarter": a["hizbQuarter"],
        })

ayah_df = pd.DataFrame(ayah_records)
ayah_df["global_order"] = range(len(ayah_df))
print(f"Surahs: {ayah_df['surah_number'].nunique()} | Ayat: {len(ayah_df)} | "
      f"Hizb quarters found: {ayah_df['hizb_quarter'].nunique()} (should be 240)")

In [ ]:
# CELL 4 -- Embed ayat (cached), build quarter and eighth vectors
ayah_embed_cache_file = EMBED_CACHE_DIR / "ayah_embeddings.pkl"

if ayah_embed_cache_file.exists():
    print("Loading cached ayah embeddings...")
    with open(ayah_embed_cache_file, "rb") as f:
        ayah_embeddings = pickle.load(f)
    print(f"Loaded {len(ayah_embeddings)} cached ayah embeddings.")
else:
    print(f"Embedding all {len(ayah_df)} ayat individually (slow, one-time only)...")
    from sentence_transformers import SentenceTransformer
    print("Loading model:", SBERT_MODEL_NAME)
    model = SentenceTransformer(SBERT_MODEL_NAME)
    all_texts = ayah_df["text"].tolist()
    all_vecs = model.encode(all_texts, batch_size=64, show_progress_bar=True,
                            normalize_embeddings=True, convert_to_numpy=True)
    ayah_embeddings = {}
    for (snum, anum), vec in zip(zip(ayah_df["surah_number"], ayah_df["ayah_number"]), all_vecs):
        ayah_embeddings[(snum, anum)] = vec
    with open(ayah_embed_cache_file, "wb") as f:
        pickle.dump(ayah_embeddings, f)
    print(f"Done and cached. {len(ayah_embeddings)} ayat embedded.")

def vector_for_ayat(ayat_keys):
    vecs = [ayah_embeddings[k] for k in ayat_keys if k in ayah_embeddings]
    return np.mean(vecs, axis=0) if vecs else None

quarter_vectors = {}
for qnum, group in ayah_df.groupby("hizb_quarter"):
    keys = list(zip(group["surah_number"], group["ayah_number"]))
    v = vector_for_ayat(keys)
    if v is not None:
        quarter_vectors[qnum] = v
print(f"Quarter vectors built: {len(quarter_vectors)} (should be 240)")

eighth_vectors = {}
for qnum, group in ayah_df.groupby("hizb_quarter"):
    group_sorted = group.sort_values("global_order")
    n = len(group_sorted)
    half = (n + 1) // 2
    first_half = group_sorted.iloc[:half]
    second_half = group_sorted.iloc[half:]
    keys_a = list(zip(first_half["surah_number"], first_half["ayah_number"]))
    keys_b = list(zip(second_half["surah_number"], second_half["ayah_number"]))
    v_a = vector_for_ayat(keys_a)
    v_b = vector_for_ayat(keys_b)
    if v_a is not None:
        eighth_vectors[f"{qnum}a"] = v_a
    if v_b is not None:
        eighth_vectors[f"{qnum}b"] = v_b
print(f"Eighth vectors built: {len(eighth_vectors)} (should be 480)")

In [ ]:
# CELL 5 -- Clustering helper
import umap, hdbscan
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score

def run_umap(matrix, n_components, n_neighbors, min_dist, metric="cosine", random_state=RANDOM_SEED):
    reducer = umap.UMAP(n_neighbors=min(n_neighbors, len(matrix) - 1), n_components=n_components,
                         min_dist=min_dist, metric=metric, random_state=random_state)
    return reducer.fit_transform(matrix)

def cluster_and_report(embeddings_dict, label_types, title, n_neighbors=UMAP_N_NEIGHBORS,
                       min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE, verbose=True):
    names = sorted(embeddings_dict.keys())
    n = len(names)
    emb_matrix = np.array([embeddings_dict[nm] for nm in names])
    umap_high = run_umap(emb_matrix, min(UMAP_N_COMPONENTS_HIGH, max(2, n - 2)), n_neighbors, 0.0, UMAP_METRIC)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min(min_cluster_size, max(2, n // 10)),
                                min_samples=HDBSCAN_MIN_SAMPLES,
                                cluster_selection_method="eom", metric="euclidean")
    labels = clusterer.fit_predict(umap_high)
    umap_2d = run_umap(emb_matrix, 2, n_neighbors, UMAP_MIN_DIST, UMAP_METRIC)
    mask = labels >= 0
    n_clusters = len(set(labels[mask])) if mask.sum() > 0 else 0
    n_outliers = int(np.sum(labels == -1))
    sil = float(silhouette_score(umap_high[mask], labels[mask])) if n_clusters >= 2 and mask.sum() > n_clusters else None
    result_df = pd.DataFrame({"name": names, "type": [label_types[nm] for nm in names],
        "cluster": labels, "umap_x": umap_2d[:, 0], "umap_y": umap_2d[:, 1]})
    if verbose:
        print(f"=== {title} ===")
        print(f"Total: {n} | Clusters: {n_clusters} | Outliers: {n_outliers} | Silhouette: {sil}")
    return result_df, {"n_entities": n, "n_clusters": n_clusters, "n_outliers": n_outliers, "silhouette": sil}

print("Clustering helper ready.")

In [ ]:
# CELL 6 -- Experiments 45 and 46: quarters and eighths, clustered alone
quarter_units = {f"Quarter {n}": v for n, v in quarter_vectors.items()}
exp45_types = {nm: "Quarter" for nm in quarter_units}
exp45_df, exp45_metrics = cluster_and_report(quarter_units, exp45_types, "Experiment 45: 240 quarters alone")
exp45_df["quarter_number"] = exp45_df["name"].str.replace("Quarter ", "").astype(int)
exp45_df = exp45_df.sort_values(["cluster", "quarter_number"])
exp45_df.to_csv(TABLES_DIR / "experiment45_full_membership.csv", index=False)

print("\nFull quarter membership by cluster:")
for cl in sorted(exp45_df["cluster"].unique()):
    members = exp45_df[exp45_df["cluster"] == cl]["quarter_number"].tolist()
    label = "Outliers" if cl == -1 else f"Cluster {cl}"
    print(f"{label} ({len(members)}): {members}")

print("\n" + "=" * 70)
eighth_units = {f"Eighth {n}": v for n, v in eighth_vectors.items()}
exp46_types = {nm: "Eighth" for nm in eighth_units}
exp46_df, exp46_metrics = cluster_and_report(eighth_units, exp46_types, "Experiment 46: 480 eighths alone")
exp46_df = exp46_df.sort_values("cluster")
exp46_df.to_csv(TABLES_DIR / "experiment46_full_membership.csv", index=False)

print("\nFull eighth membership by cluster:")
for cl in sorted(exp46_df["cluster"].unique()):
    members = exp46_df[exp46_df["cluster"] == cl]["name"].tolist()
    label = "Outliers" if cl == -1 else f"Cluster {cl}"
    print(f"{label} ({len(members)}): {members}")

In [ ]:
# CELL 7 -- Figures
def plot_alone(df, title, save_name, marker):
    fig, ax = plt.subplots(figsize=(11, 8.5))
    unique_clusters = sorted(df["cluster"].unique())
    n_clust_plot = len([c for c in unique_clusters if c >= 0])
    colors = plt.cm.tab20(np.linspace(0, 1, max(n_clust_plot, 1)))
    for cl in unique_clusters:
        sub = df[df["cluster"] == cl]
        color = "gray" if cl == -1 else colors[cl % len(colors)]
        ax.scatter(sub["umap_x"], sub["umap_y"], c=[color], marker=marker, s=70,
                  alpha=0.85, edgecolors="black", linewidth=0.4)
    ax.set_xlabel("UMAP Dimension 1"); ax.set_ylabel("UMAP Dimension 2")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / save_name, dpi=300, bbox_inches="tight")
    plt.show()

plot_alone(exp45_df, "Experiment 45: 240 Quarters Alone", "experiment45_umap.png", "^")
plot_alone(exp46_df, "Experiment 46: 480 Eighths Alone", "experiment46_umap.png", "v")
print("Figures saved.")

In [ ]:
# CELL 8 -- Final report
report_path = REPORTS_DIR / "experiments_45_46_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("=" * 70 + "\n")
    f.write("EXPERIMENTS 45-46: QUARTERS AND EIGHTHS, CLUSTERED ALONE\n")
    f.write("=" * 70 + "\n\n")

    f.write("EXPERIMENT 45: 240 QUARTERS ALONE\n" + "-" * 40 + "\n")
    for k, v in exp45_metrics.items():
        f.write(f"  {k}: {v}\n")
    f.write("  Full membership by cluster:\n")
    for cl in sorted(exp45_df["cluster"].unique()):
        members = exp45_df[exp45_df["cluster"] == cl]["quarter_number"].tolist()
        label = "Outliers" if cl == -1 else f"Cluster {cl}"
        f.write(f"    {label} ({len(members)}): {members}\n")

    f.write("\nEXPERIMENT 46: 480 EIGHTHS ALONE\n" + "-" * 40 + "\n")
    for k, v in exp46_metrics.items():
        f.write(f"  {k}: {v}\n")
    f.write("  Full membership by cluster:\n")
    for cl in sorted(exp46_df["cluster"].unique()):
        members = exp46_df[exp46_df["cluster"] == cl]["name"].tolist()
        label = "Outliers" if cl == -1 else f"Cluster {cl}"
        f.write(f"    {label} ({len(members)}): {members}\n")

print(f"Report written to {report_path}")
print()
print(open(report_path, encoding="utf-8").read())

## Done

Output in `output/`:
- `output/tables/experiment45/46_full_membership.csv`
- `output/figures/experiment45/46_umap.png`
- `output/reports/experiments_45_46_report.txt`

Send this back and I'll write the short report.